# 심화 미션: 스마트팜 온실 출하 기록
- 상황: 선별대에서 도장을 찍기 전에 등외를 미리 알고 싶다
- 목표: 오늘 배운 순서를 다른 데이터로 혼자 한 바퀴 돌린다

### 용어 풀이 - 온실에서 쓰는 말

| 말 | 뜻 |
|---|---|
| 상품 / 등외 | 선별대에서 붙이는 판정. 등외는 제값에 못 파는 것 |
| 양액 (EC) | 물에 녹인 거름. 그 진하기를 dS/m 라는 단위로 잰다 |
| 산도 (pH) | 산성인지 알칼리성인지. 7이 중간이고 낮을수록 산성 |
| 토양 수분 | 흙에 물기가 얼마나 있는지 (%) |
| 야간 최저 기온 | 밤에 가장 낮았던 기온. 작물이 스트레스를 받는 지점 |

---

## Q1. 파일 열고 크기 확인하기

In [11]:
import pandas as pd

df = pd.read_csv("../../data/day03_greenhouse.csv")

print(df.shape)
print(df["result"].value_counts())

(2000, 13)
result
상품    1861
등외     139
Name: count, dtype: int64


---
## Q2. 등외는 얼마나 드문가

In [12]:
개수 = df["result"].value_counts()
비율 = df["result"].value_counts(normalize=True) * 100

print("상품:", 개수["상품"], "건")
print("등외:", 개수["등외"], "건")
print()
print("상품 비율:", round(비율["상품"], 2), "%")
print("등외 비율:", round(비율["등외"], 2), "%")

상품: 1861 건
등외: 139 건

상품 비율: 93.05 %
등외 비율: 6.95 %


---
## Q3. 정답표를 숫자로 바꾸기

In [13]:
# result 열은 "상품"·"등외"라는 글자다. 상품은 0, 등외는 1로 바꾼다
df["등외여부"] = (df["result"] == "등외").astype(int)

print(df["등외여부"].value_counts())

등외여부
0    1861
1     139
Name: count, dtype: int64


---
## Q4. 입력과 정답으로 가르기

In [14]:
# 입력 - 센서 측정값 열만. 식별자(batch_id, harvested_at, house_id, crop)와
# 정답표(result, 등외여부)는 절대 들어가면 안 된다
제외열 = ["batch_id", "harvested_at", "house_id", "crop", "result", "등외여부"]
센서열 = [c for c in df.columns if c not in 제외열]
X = df[센서열]

# 정답 - 맞혀야 할 것
y = df["등외여부"]

print("입력 열:", 센서열)
print("입력:", X.shape)
print("정답:", y.shape)

입력 열: ['temp_avg', 'humidity_avg', 'co2_ppm', 'soil_moisture', 'ec', 'ph', 'light_hours', 'night_temp_min']
입력: (2000, 8)
정답: (2000,)


---
## Q5. 학습용과 시험용으로 나누기

In [15]:
# train_test_split - 표를 학습용과 시험용 두 몫으로 갈라준다
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 시험용으로 떼어둘 비율 (20%)
    random_state=42,      # 무작위로 섞되, 다시 실행해도 같게 나오도록 고정
    stratify=y            # 불량 비율을 양쪽에 똑같이 맞춰서 나눈다 (층화추출)
)

print(f"학습용: {X_train.shape[0]} | 등외 {int(y_train.sum())} ({y_train.mean() * 100:.2f}%)")
print(f"시험용: {X_test.shape[0]} | 등외 {int(y_test.sum())} ({y_test.mean() * 100:.2f}%)")

학습용: 1600 | 등외 111 (6.94%)
시험용: 400 | 등외 28 (7.00%)


---
## Q6. 아무것도 배우지 않은 기준 모델